# Step 1a 확장: RTMS 전체 이력 수집 (초기 1회용)

**목적**: 2006년 1월 ~ 현재까지 전체 데이터 초기 적재.

**호출 규모**: 245개월 × 66개 시군구 = 약 16,170회

**일 호출 한도**: 10,000건 → 이틀에 나눠서 실행
- Day 1: 2006년 1월 ~ 2016년 6월 (약 126개월 × 66 = 8,316회)
- Day 2: 2016년 7월 ~ 현재 (약 119개월 × 66 = 7,854회)

**이미 수집된 파일은 건너뜀** — 중단됐다가 재실행해도 안전

## 0. 환경 로드

In [ ]:
import sys
sys.path.insert(0, r'C:\projects\realestate_reco')  # 본인 경로
from common import *

import time
import requests
import xml.etree.ElementTree as ET
from datetime import datetime
import pandas as pd
from dateutil.relativedelta import relativedelta

SECRETS = load_secrets()
SERVICE_KEY = SECRETS['data_go_kr_service_key']
MOCK_MODE = SECRETS.get('MOCK_MODE', False)

print(f'MOCK_MODE = {MOCK_MODE}')
print(f'시군구 수 = {len(SIGUNGU_CODES)}')

## 1. 수집 대상 월 목록 생성

In [ ]:
def generate_ym_list(start_ym: str, end_ym: str) -> list:
    """start_ym ~ end_ym 사이 모든 월 목록 생성.
    
    start_ym: '200601'
    end_ym:   '202604'
    """
    start = datetime.strptime(start_ym, '%Y%m')
    end   = datetime.strptime(end_ym,   '%Y%m')
    result = []
    cur = start
    while cur <= end:
        result.append(cur.strftime('%Y%m'))
        cur += relativedelta(months=1)
    return result


# 전체 기간 설정
START_YM = '200601'  # RTMS 데이터 시작
END_YM   = datetime.now().strftime('%Y%m')  # 오늘 기준 이번 달

all_yms = generate_ym_list(START_YM, END_YM)
print(f'수집 대상: {START_YM} ~ {END_YM}')
print(f'총 {len(all_yms)}개월 × {len(SIGUNGU_CODES)}개 시군구 = {len(all_yms)*len(SIGUNGU_CODES):,}회 호출 예정')
print(f'\n예상 소요 시간: 약 {len(all_yms)*len(SIGUNGU_CODES)*0.35/3600:.1f}시간 (호출당 0.35초 기준)')

## 2. API 호출 함수 (step1a와 동일)

In [ ]:
RTMS_TRADE_URL = 'http://openapi.molit.go.kr/OpenAPI_ToolInstallPackage/service/rest/RTMSOBJSvc/getRTMSDataSvcAptTradeDev'

def fetch_rtms_trade(sigungu_code: str, ym: str, timeout: int = 30) -> str:
    if MOCK_MODE:
        return f'''<?xml version="1.0" encoding="UTF-8"?>
<response><header><resultCode>00</resultCode></header>
<body><items><item>
  <거래금액>100,000</거래금액><건축년도>2010</건축년도>
  <년>{ym[:4]}</년><월>{int(ym[4:])}</월><일>15</일>
  <법정동>테스트동</법정동><아파트>테스트단지</아파트>
  <전용면적>84.5</전용면적><지번>100</지번>
  <지역코드>{sigungu_code}</지역코드><층>5</층>
  <해제여부></해제여부>
</item></items><totalCount>1</totalCount></body></response>'''

    params = {
        'serviceKey': SERVICE_KEY,
        'LAWD_CD': sigungu_code,
        'DEAL_YMD': ym,
        'numOfRows': 1000,
        'pageNo': 1,
    }
    resp = requests.get(RTMS_TRADE_URL, params=params, timeout=timeout)
    resp.raise_for_status()
    return resp.text


def parse_row_count(xml_text: str) -> int:
    """XML에서 totalCount만 빠르게 추출."""
    try:
        root = ET.fromstring(xml_text)
        tc = root.find('.//totalCount')
        return int(tc.text) if tc is not None else 0
    except:
        return -1

## 3. 전체 이력 수집 함수

**핵심**: 이미 수집된 파일은 건너뜀 → 중간에 멈춰도 재실행하면 이어서 진행

In [ ]:
def collect_history(
    ym_list: list,
    skip_existing: bool = True,
    sleep_per_call: float = 0.35,
    retries: int = 3,
) -> pd.DataFrame:
    """주어진 월 목록 × 수도권 전체 시군구 수집.

    skip_existing=True: 이미 xml 파일이 있으면 건너뜀 (재실행 안전)
    """
    total = len(ym_list) * len(SIGUNGU_CODES)
    done = 0
    skipped = 0
    errors = []
    start_time = time.time()

    for ym in ym_list:
        year, month = ym[:4], ym[4:]
        out_dir = DIRS['raw_trade'] / year / month
        out_dir.mkdir(parents=True, exist_ok=True)

        for code, name in SIGUNGU_CODES.items():
            xml_path  = out_dir / f'{code}_{ym}.xml'
            meta_path = out_dir / f'{code}_{ym}.meta.json'

            # 이미 수집된 파일이면 건너뜀
            if skip_existing and xml_path.exists():
                skipped += 1
                done += 1
                continue

            last_err = None
            for attempt in range(1, retries + 1):
                try:
                    xml_text = fetch_rtms_trade(code, ym)
                    rows = parse_row_count(xml_text)
                    xml_path.write_text(xml_text, encoding='utf-8')
                    meta_path.write_text(
                        json.dumps({
                            'sigungu_code': code, 'sigungu_name': name,
                            'ym': ym, 'rows': rows,
                            'fetched_at': datetime.now().isoformat(),
                            'mock': MOCK_MODE,
                        }, ensure_ascii=False, indent=2),
                        encoding='utf-8'
                    )
                    last_err = None
                    time.sleep(sleep_per_call)
                    break
                except Exception as e:
                    last_err = e
                    time.sleep(2 ** attempt)

            if last_err:
                errors.append({'ym': ym, 'code': code, 'name': name, 'error': str(last_err)})

            done += 1

            # 진행률 표시 (100건마다)
            if done % 100 == 0:
                elapsed = time.time() - start_time
                remaining = elapsed / done * (total - done)
                print(
                    f'[{done:,}/{total:,}] '
                    f'현재: {ym} {name} | '
                    f'건너뜀: {skipped} | '
                    f'오류: {len(errors)} | '
                    f'남은시간: {remaining/60:.0f}분'
                )

    print(f'\n완료: {done:,}건 처리 / 건너뜀: {skipped} / 오류: {len(errors)}')
    return pd.DataFrame(errors) if errors else pd.DataFrame()

## 4. Day 1 실행: 2006년 1월 ~ 2016년 6월

약 8,316회 호출. 예상 소요 시간: **약 50분**

완료 후 내일 Day 2 실행.

In [ ]:
day1_yms = generate_ym_list('200601', '201606')
print(f'Day 1 대상: {day1_yms[0]} ~ {day1_yms[-1]} ({len(day1_yms)}개월)')
print(f'호출 수: {len(day1_yms) * len(SIGUNGU_CODES):,}회')

df_errors_day1 = collect_history(day1_yms)

if len(df_errors_day1) > 0:
    print('\n오류 목록:')
    print(df_errors_day1)

## 5. Day 2 실행: 2016년 7월 ~ 현재

Day 1 완료 다음날 실행. 약 7,854회 호출. 예상 소요 시간: **약 46분**

In [ ]:
day2_yms = generate_ym_list('201607', END_YM)
print(f'Day 2 대상: {day2_yms[0]} ~ {day2_yms[-1]} ({len(day2_yms)}개월)')
print(f'호출 수: {len(day2_yms) * len(SIGUNGU_CODES):,}회')

df_errors_day2 = collect_history(day2_yms)

if len(df_errors_day2) > 0:
    print('\n오류 목록:')
    print(df_errors_day2)

## 6. 수집 결과 확인

In [ ]:
# 수집된 XML 파일 수 확인
xml_files = list(DIRS['raw_trade'].rglob('*.xml'))
meta_files = list(DIRS['raw_trade'].rglob('*.meta.json'))

print(f'수집된 XML 파일: {len(xml_files):,}개')
print(f'메타 파일:       {len(meta_files):,}개')
print(f'예상 파일 수:    {len(all_yms) * len(SIGUNGU_CODES):,}개')
print(f'\n누락 파일: {len(all_yms) * len(SIGUNGU_CODES) - len(xml_files):,}개')

# 메타 파일에서 총 거래 건수 집계
total_rows = 0
for meta_path in meta_files:
    try:
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        total_rows += meta.get('rows', 0)
    except:
        pass

print(f'총 수집 거래 건수: {total_rows:,}건')

## 7. 누락 파일 재수집 (오류 발생 시)

위 확인에서 누락 파일이 있으면 아래 셀로 재수집.

In [ ]:
def find_missing_and_recollect():
    """누락된 (ym, sigungu) 조합 찾아서 재수집."""
    missing = []
    for ym in all_yms:
        year, month = ym[:4], ym[4:]
        out_dir = DIRS['raw_trade'] / year / month
        for code in SIGUNGU_CODES:
            xml_path = out_dir / f'{code}_{ym}.xml'
            if not xml_path.exists():
                missing.append((ym, code))

    print(f'누락: {len(missing)}개')
    if not missing:
        return

    for ym, code in missing:
        year, month = ym[:4], ym[4:]
        out_dir = DIRS['raw_trade'] / year / month
        out_dir.mkdir(parents=True, exist_ok=True)
        xml_path  = out_dir / f'{code}_{ym}.xml'
        meta_path = out_dir / f'{code}_{ym}.meta.json'
        try:
            xml_text = fetch_rtms_trade(code, ym)
            rows = parse_row_count(xml_text)
            xml_path.write_text(xml_text, encoding='utf-8')
            meta_path.write_text(
                json.dumps({'sigungu_code': code, 'ym': ym, 'rows': rows,
                            'fetched_at': datetime.now().isoformat()},
                           ensure_ascii=False), encoding='utf-8')
            print(f'  재수집 완료: {ym} {code}')
            time.sleep(0.35)
        except Exception as e:
            print(f'  재수집 실패: {ym} {code} → {e}')

# find_missing_and_recollect()  # 누락 있을 때만 주석 해제

---

## 다음 단계

전체 수집 완료 후:
1. `step2_clean_apt_id_map.ipynb` — raw XML 전체 정제 (시간 오래 걸림)
2. `step3_build_master.ipynb` — master 테이블 빌드
3. 이후부터는 `step1a` → `step2` → `step3` 매일 실행 (직전 3개월만 재수집)